# Experiments 51 -  ***OPTUNA SEARCH***
Weight testing for the loss function *(box/dfl/cls)*.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v4i)***
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. Optuna hyperparameter search: `cls=1~5` | `dfl=0.5~3` | `box=1~7`
- **Reference:** Default parameters: `cls=0.5` | `dfl=1.5` | `box=7.5`


## Init

In [ ]:
import os
import shutil
import fnmatch
import pickle

In [ ]:
!pip install optuna

In [ ]:
!pip install ultralytics

## Helper Functions

In [ ]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [ ]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [ ]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [ ]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [ ]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [ ]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

### Graph functions

In [ ]:
def show_cm(TP, FP, FN):
    matrix = [[TP, FP], [FN, 0]]
    total_det = sum(sum(value) for value in matrix)
    percentages = []
    for row in matrix:
        values_percentages = []
        for value in row:
            if total_det != 0:
                percentage = (value / total_det) * 100
            else:
                percentage = 0.0
            values_percentages.append(f"{percentage:.2f}%")
        percentages.append(values_percentages)
    
    print("Total objects detected:", total_det)
    print("\nConfusion matrix:")
    for row in percentages:
        a, b = row
        print(f"[ {a} , {b} ]")

In [ ]:
def show_metrics(TP, FP, FN):
    show_cm(TP, FP, FN)
    accuracy = TP/(TP+FP+FN)
    precision = TP/(TP+FP)
    recall = TP/(TP+FN)
    f1 = 2 * (precision * recall) / (precision + recall)
    f2 = 1.25 * (precision * recall) / (0.25 * precision + recall)
    fm = (precision * recall) ** 0.5
    print("\nMetrics:")
    print(f"- Accuracy: {accuracy:.3f}")
    print(f"- Precision: {precision:.3f}")
    print(f"- Recall: {recall:.3f}")
    print(f"- F1 Score: {f1:.3f}")
    print(f"- F½ Score: {f2:.3f}")
    print(f"- G-mean: {fm:.3f}")

In [ ]:
cm = lambda results: results.confusion_matrix.matrix.tolist() if hasattr(results, 'confusion_matrix') and hasattr(results.confusion_matrix, 'matrix') else None

In [ ]:
import json

def save_json(results):
  try:
    data_to_store = {
        "confusion_matrix": cm(results),
        "results_dict": results.results_dict,
        "speed": results.speed
    }

    # Convert the Python dictionary to a JSON string
    json_data = json.dumps(data_to_store, indent=4)

    # You can now save this JSON string to a file
    folder = str(results.save_dir)
    with open(f"/content/{folder}/results.json", "w") as f:
        f.write(json_data)

    print("✅ JSON file stored in:", folder)

  except Exception as e:
      print(f"❌ An error occurred: {e}")

  #return json_data


In [ ]:
def gimme_metrics(results):
  matrix = cm(results)
  total_det = sum(sum(value) for value in matrix)
  percentages = []
  for row in matrix:
      values_percentages = []
      for value in row:
          if total_det != 0:
              percentage = (value / total_det) * 100
          else:
              percentage = 0.0
          values_percentages.append(f"{percentage:.2f}%")
      percentages.append(values_percentages)

  print("Total objects detected:", total_det)
  print("Confusion matrix:")
  for row in percentages:
      print(row)
      
  return matrix


In [ ]:
def numoji(numero):
  """
  Convierte un número entero del 1 al 10 a su emoji correspondiente.

  Args:
    numero: Un entero entre 1 y 10.

  Returns:
    Un string con el emoji correspondiente al número, o "0️⃣" si el número
    está fuera del rango.
  """
  if 0 <= numero <= 10:
    emoji_map = {
        0: "0️⃣",
        1: "1️⃣",
        2: "2️⃣",
        3: "3️⃣",
        4: "4️⃣",
        5: "5️⃣",
        6: "6️⃣",
        7: "7️⃣",
        8: "8️⃣",
        9: "9️⃣",
        10: "🔟"
    }
    return emoji_map[numero]
  else:
    return "*️⃣"

# Datasets builder

## Importing from Drive

In [ ]:
!rm -rf /content/sample_data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

In [ ]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

In [ ]:
choose_dataset = 5
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [ ]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [ ]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

## Download model

In [ ]:
from ultralytics import YOLO

In [ ]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

# Finetuning

### Training optimization

In [ ]:
# Libera memoria de la GPU en caso de OOM error
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [ ]:
!nvidia-smi

In [ ]:
!yolo version

-----
## Experiment 51
### *Full fine-tuned (no freeze) | Hyperparameters serach*
Applying optuna for finding best values for weighted loss function

In [ ]:
average_time = 12 # average time per optuna experiment measured earlier

### Train experiment


    1. Optuna hyperparameter search: `cls=1~5` | `dfl=0.5~3` | `box=1~7`

In [ ]:
import optuna

def objective(trial):

    # Suggest values for conf and iou
    cls_threshold = trial.suggest_float("cls", 1, 5) # Define a reasonable range
    dfl_threshold = trial.suggest_float("dfl", 0.5, 2)   # Define a reasonable range
    box_threshold = trial.suggest_float("box", 1, 7)   # Define a reasonable range
    emoji_number = "".join([numoji(int(i)) for i in str(trial.number)])
    print(f"\n\n{emoji_number} Trial {trial.number}: Trying cls={cls_threshold:.4f}, dfl={dfl_threshold:.4f}, box={box_threshold:.4f}")

    try:
        # Run training with the suggested hyperparameters
        # Higher time and lower patience will allow the model to train longer
        # and potentially find better hyperparameters, without sacrificing performance.
    
        # 1. TRAINING
        # Load the pretrained YOLO model
        model = YOLO("yolov8m.pt")

        history = model.train(
            data=data,
            epochs=150, # ~ ideally 10 epoch/minute
            val=False, # No validation during training
            imgsz=640,
            batch=-1,
            patience=10, # Early stopping patience
            time = 0.5, # Every experiment will train for 30 minutes
            # =============================================
            # Values tested on this experiment
            cls=cls_threshold, # Class confidence threshold
            dfl=dfl_threshold, # Distribution Focal Loss
            box=box_threshold, # Box confidence threshold
            # =============================================
        ) # As long as time is limited, epochs will adjust automatically to fit the time limit.
        print(f"Saved into: {history.save_dir}")


        # 2. VALIDATION
        # Load currently trained YOLO model
        model_val = YOLO(f"/content/{history.save_dir}/weights/best.pt")

        results = model_val.val(
            data=data,
            batch=-1,
            verbose=True, # Keep verbose to see detailed output
            save_json=True # Save JSON
        )
        print(f"Saved into: {results.save_dir}")

        # VERIFICATION
        # Checks metrics for the current trial & store best model

        # Display confusion matrix and store results as JSON
        print()
        gimme_metrics(results)
        print()
        save_json(results)
        print()

        # Extract the F1-Score.
        if hasattr(results, 'results_dict') and results.results_dict is not None:
          precision = results.results_dict['metrics/precision(B)']
          recall = results.results_dict['metrics/recall(B)']

          if precision is not None and recall is not None and (precision + recall) > 0:
              f1_score = 2 * (precision * recall) / (precision + recall)
              print(f"Trial {trial.number}: Calculated F1@0.5 = {f1_score:.4f} (P={precision:.4f}, R={recall:.4f})")
          else:
              print(f"❌ Trial {trial.number}: Could not find/calculate F1@0.5, precision@0.5 or recall@0.5 in results.metrics.")
              f1_score = 0.0 # Assign a low score if metric is missing
        else:
          print(f"❌ Trial {trial.number}: Could not access metrics from results object directly.")
          f1_score = 0.0 # Return 0.0 if metrics cannot be accessed

        # STORE RESULTS
        # Save the model with the best F1-Score

        print("="*50)
        return f1_score

    except Exception as e:
        print(f"Trial {trial.number}: An error occurred during validation: {e}")
        # Retornar un valor bajo para indicar que este conjunto de hiperparámetros
        # probablemente no es bueno o causó un error.
        return 0.0

In [ ]:
import logging
import sys
#import os

# Optional: Configure logging for Optuna
logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# Define the storage path for the Optuna study
# Using a local SQLite database file
db_path = "sqlite:///optuna_yolov8_f1_study.db"
study_name = "yolov8_loss_optimization"

# The study progress is automatically saved to 'optuna_yolov8_f1_study.db'
# in the same directory where you run the script.

In [ ]:
# Option for setting max n_trials with GPU resources
gpu_limit = 1
n_trials = round(gpu_limit*3600/average_time)

print(f"Starting training for {gpu_limit} hour limit")
print(f"Optuna will run {n_trials}")

In [ ]:
import time

# Define manually the number of trials to do (optional)
# n_trials = 3 # You can start with a small number, e.g., 50 or 100

# --- Study Creation ---
# Check if the study already exists. If so, load it; otherwise, create a new one.
# This allows resuming the optimization later.
try:
    # Load the existing study
    study = optuna.load_study(study_name=study_name, storage=db_path)
    print(f"Resuming existing study '{study_name}' from {db_path}")
except KeyError:
    # Create a new study if it doesn't exist
    study = optuna.create_study(study_name=study_name, storage=db_path, direction="maximize")
    print(f"Created a new study '{study_name}' at {db_path}")

# --- Study Execution ---
print(f"Running Optuna optimization for F1-score ({n_trials} trials)...")
# Measuring experiment time
start_time = time.perf_counter_ns()

# Run the optimization
# Increase n_trials for a more exhaustive search.
study.optimize(objective, n_trials=n_trials)

# Stop time measurment
end_time = time.perf_counter_ns()

# --- Study Results Analytics ---
# Display the best hyperparameters and the best F1 value found
print("\nOptimization finished.")
print("Best hyperparameters: ", study.best_params)
print("Best F1-Score: ", study.best_value)
print()

# You can access the best trial if you need more details
best_trial = study.best_trial
print(f"Best trial: Number {best_trial.number}, Value {best_trial.value}")
print("Hyperparameters of the best trial: ", best_trial.params)

elapsed_time_ns = end_time - start_time
elapsed_time_s = elapsed_time_ns / 1e9
average_time = elapsed_time_s/n_trials

print(f"\n\nElapsed time: {elapsed_time_s:.2f} seconds for {n_trials}")
print(f"Average time: {average_time:.3f} seconds")

## Extract results for best hyperparams

In [ ]:
# Validate the model with Optuna's best parameters
best_conf = study.best_params['conf']
best_iou = study.best_params['iou']

results = model.val(data="/content/YOLO/3.5m.v3i.yolov8.640px/data.yaml",
          batch=64,
          conf=best_conf,
          iou=best_iou,
          verbose=True,
          save_json=True)

In [ ]:
print(best_conf, best_iou)

### Metrics

In [ ]:
gimme_metrics(results)

In [ ]:
save_json(results)

## Save all results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/optuna_yolov8_f1_study.db', destination='/content/drive/MyDrive/save/')

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save/')